# Python - OpenStudio 조작 테스트

## openstudio 설치 테스트   
- pip install openstudio==3.11.0

In [1]:
import openstudio # OpenStudio 모듈
from pathlib import Path # 파일 경로 처리를 위한 pathlib 모듈

In [2]:
print(openstudio.openStudioVersion())

3.11.0


In [3]:
# 불러올 OSM 파일 경로 설정
osm_path = Path(r"C:\Users\ryudo\OneDrive\Desktop\SampleBuilding\SampleBuilding_v2.osm")

# OSM 파일을 OpenStudio Path 형식으로 변환
os_path = openstudio.path(str(osm_path))

# OSM 파일 불러오기
loaded = openstudio.osversion.VersionTranslator().loadModel(os_path)

if not loaded.is_initialized():
    raise ValueError("OSM 파일을 불러오지 못했습니다.")

# OptionalModel에서 실제 Model 객체 꺼내기
model = loaded.get()

# 모델 내 공간/존/표면 개수 확인
print("OpenStudio 버전:", openstudio.openStudioVersion())
print("Space 개수:", len(model.getSpaces()))
print("Thermal Zone 개수:", len(model.getThermalZones()))
print("Surface 개수:", len(model.getSurfaces()))

OpenStudio 버전: 3.11.0
Space 개수: 23
Thermal Zone 개수: 15
Surface 개수: 207


## 1. Thermal zones 이름 변경하기

In [4]:
# Thermal Zone 불러오기
zones = model.getThermalZones()

# 모든 Thermal Zone의 이름을 변경 
## zone-n -> zone-n_revised
if zones:
    for zone in zones:
        old_name = zone.nameString()
        new_name = old_name + "_revised"
        zone.setName(new_name)
        print(f"존 이름 변경: '{old_name}' -> '{new_name}'")
    print(f"존 이름 변경 완료. 총 {len(zones)}개의 Thermal Zone 이름이 변경되었습니다.")
    
else:
    print("모델에 Thermal Zone이 없습니다.")

존 이름 변경: 'zone-1' -> 'zone-1_revised'
존 이름 변경: 'zone-13' -> 'zone-13_revised'
존 이름 변경: 'zone-10' -> 'zone-10_revised'
존 이름 변경: 'zone-11' -> 'zone-11_revised'
존 이름 변경: 'zone-15' -> 'zone-15_revised'
존 이름 변경: 'zone-4' -> 'zone-4_revised'
존 이름 변경: 'zone-12' -> 'zone-12_revised'
존 이름 변경: 'zone-14' -> 'zone-14_revised'
존 이름 변경: 'zone-2' -> 'zone-2_revised'
존 이름 변경: 'zone-3' -> 'zone-3_revised'
존 이름 변경: 'zone-7' -> 'zone-7_revised'
존 이름 변경: 'zone-5' -> 'zone-5_revised'
존 이름 변경: 'zone-6' -> 'zone-6_revised'
존 이름 변경: 'zone-8' -> 'zone-8_revised'
존 이름 변경: 'zone-9' -> 'zone-9_revised'
존 이름 변경 완료. 총 15개의 Thermal Zone 이름이 변경되었습니다.


## 2. 외부 벽채 재료 변경

In [5]:
import math
from collections import defaultdict

# model = openstudio.osversion.VersionTranslator().loadModel(
#     openstudio.path(str(osm_path))
# ).get()

In [6]:
# 모든 surfaceType 목록 출력
types = set(s.surfaceType() for s in model.getSurfaces())
print("surfaceType 종류:", types)

surfaceType 종류: {'Wall', 'RoofCeiling', 'Floor'}


In [7]:
# 모든 벽 종류 출력하기
for surface in model.getSurfaces():
    if surface.surfaceType() == "Wall": # RoofCeiling, Wall, Floor 
        print(f"{surface.surfaceType():10s} | {surface.nameString()}")

Wall       | su-w-25-e-w-10
Wall       | su-e-28-29-i-w-31 Reversed
Wall       | su-e-29-e-w-38
Wall       | su-s-24-29-i-w-6 Reversed
Wall       | su-s-29-30-i-w-40 Reversed
Wall       | su-s-29-30-i-w-40
Wall       | su-e-40-45-i-w-109 Reversed
Wall       | su-e-25-28-i-w-13
Wall       | su-e-45-e-w-127
Wall       | su-s-44-45-i-w-126 Reversed
Wall       | su-w-31-e-w-49
Wall       | su-n-27-31-i-w-26 Reversed
Wall       | su-s-45-46-i-w-129
Wall       | su-e-24-e-w-2
Wall       | su-n-24-e-w-1
Wall       | su-s-24-29-i-w-6
Wall       | su-w-33-34-i-w-73 Reversed
Wall       | su-s-24-31-i-w-7
Wall       | su-e-43-46-i-w-121 Reversed
Wall       | su-w-24-28-i-w-4
Wall       | su-w-24-28-i-w-5
Wall       | su-e-38-44-i-w-100
Wall       | su-s-31-43-i-w-61 Reversed
Wall       | su-n-31-38-i-w-56 Reversed
Wall       | su-s-38-40-i-w-99
Wall       | su-n-38-e-w-95
Wall       | su-s-45-46-i-w-129 Reversed
Wall       | su-n-34-e-w-76
Wall       | su-s-39-41-i-w-105 Reversed
Wall       | su-

In [8]:
# Wall 이름 변경: {space명}-Wall {n} (아래서부터 시계 반대 방향)
for space in model.getSpaces():
    space_name = space.nameString()
    walls = [s for s in space.surfaces() if s.surfaceType() == "Wall"]

    if not walls:
        continue

    # space 중심 (벽 centroid 평균)
    cx = sum(w.centroid().x() for w in walls) / len(walls)
    cy = sum(w.centroid().y() for w in walls) / len(walls)

    def sort_key(wall):
        dx = wall.centroid().x() - cx
        dy = wall.centroid().y() - cy
        # atan2: 0=동, π/2=북, -π/2=남(아래)
        angle = math.atan2(dy, dx)
        # 남쪽(아래, -π/2)을 0으로 시프트 → 시계 반대 방향 정렬
        angle = angle + math.pi / 2
        if angle < 0:
            angle += 2 * math.pi
        return angle

    walls_sorted = sorted(walls, key=sort_key)

    for i, wall in enumerate(walls_sorted, 1):
        suffix = " Reversed" if wall.adjacentSurface().is_initialized() else ""
        new_name = f"{space_name}-Wall {i:02d}{suffix}"
        wall.setName(new_name)
        # print(f"{new_name}")
print("Wall 이름 변경 완료.")

Wall 이름 변경 완료.


In [9]:
# Floor 이름 변경: {space명}-Floor {n} (아래서부터 시계 반대 방향)
for space in model.getSpaces():
    space_name = space.nameString()
    floors = [s for s in space.surfaces() if s.surfaceType() == "Floor"]

    if not floors:
        continue

    # space 중심 (벽 centroid 평균)
    cx = sum(f.centroid().x() for f in floors) / len(floors)
    cy = sum(f.centroid().y() for f in floors) / len(floors)

    def sort_key(floor):
        dx = floor.centroid().x() - cx
        dy = floor.centroid().y() - cy
        # atan2: 0=동, π/2=북, -π/2=남(아래)
        angle = math.atan2(dy, dx)
        # 남쪽(아래, -π/2)을 0으로 시프트 → 시계 반대 방향 정렬
        angle = angle + math.pi / 2
        if angle < 0:
            angle += 2 * math.pi
        return angle

    floors_sorted = sorted(floors, key=sort_key)

    for i, floor in enumerate(floors_sorted, 1):
        suffix = " Reversed" if floor.adjacentSurface().is_initialized() else ""
        new_name = f"{space_name}-Floor {i:02d}{suffix}"
        floor.setName(new_name)
        # print(f"{new_name}")
print("Floor 이름 변경 완료.")

Floor 이름 변경 완료.


In [10]:
# RoofCeiling 이름 변경: {space명}-RoofCeiling {n} (아래서부터 시계 반대 방향)
for space in model.getSpaces():
    space_name = space.nameString()
    roofceilings = [s for s in space.surfaces() if s.surfaceType() == "RoofCeiling"]

    if not roofceilings:
        continue

    # space 중심 (벽 centroid 평균)
    cx = sum(r.centroid().x() for r in roofceilings) / len(roofceilings)
    cy = sum(r.centroid().y() for r in roofceilings) / len(roofceilings)

    def sort_key(roofceiling):
        dx = roofceiling.centroid().x() - cx
        dy = roofceiling.centroid().y() - cy
        # atan2: 0=동, π/2=북, -π/2=남(아래)
        angle = math.atan2(dy, dx)
        # 남쪽(아래, -π/2)을 0으로 시프트 → 시계 반대 방향 정렬
        angle = angle + math.pi / 2
        if angle < 0:
            angle += 2 * math.pi
        return angle

    roofceilings_sorted = sorted(roofceilings, key=sort_key)

    for i, roofceiling in enumerate(roofceilings_sorted, 1):
        suffix = " Reversed" if roofceiling.adjacentSurface().is_initialized() else ""
        new_name = f"{space_name}-RoofCeiling {i:02d}{suffix}"
        roofceiling.setName(new_name)
        # print(f"{new_name}")
print("RoofCeiling 이름 변경 완료.")

RoofCeiling 이름 변경 완료.


In [11]:
for surface in model.getSurfaces():
    if surface.surfaceType() == "Wall": # RoofCeiling, Wall, Floor 
        print(f"{surface.surfaceType():10s} | {surface.nameString()}")

Wall       | Space 7-Wall 05
Wall       | Space 2-Wall 04 Reversed
Wall       | Space 2-Wall 02
Wall       | Space 2-Wall 03 Reversed
Wall       | Space 3-Wall 03 Reversed
Wall       | Space 2-Wall 01 Reversed
Wall       | Space 20-Wall 04 Reversed
Wall       | Space 7-Wall 03 Reversed
Wall       | Space 20-Wall 02
Wall       | Space 20-Wall 03 Reversed
Wall       | Space 11-Wall 17
Wall       | Space 11-Wall 20 Reversed
Wall       | Space 20-Wall 01 Reversed
Wall       | Space 1-Wall 03
Wall       | Space 1-Wall 04
Wall       | Space 1-Wall 01 Reversed
Wall       | Space 10-Wall 02 Reversed
Wall       | Space 1-Wall 02 Reversed
Wall       | Space 23-Wall 05 Reversed
Wall       | Space 1-Wall 05 Reversed
Wall       | Space 1-Wall 06 Reversed
Wall       | Space 16-Wall 03 Reversed
Wall       | Space 22-Wall 04 Reversed
Wall       | Space 16-Wall 02 Reversed
Wall       | Space 16-Wall 01 Reversed
Wall       | Space 16-Wall 04
Wall       | Space 23-Wall 04 Reversed
Wall       | Space 10-W

이름 변경 완료

##  Thermal zone 재설정
- 동일 층계, 동일 방향에 있는 space 묶기
- 모서리는 하나의 zone으로 
- 내부도 하나의 zone으로 

벽 방향, 층별로 출력하기

In [12]:
# 벽 방향과 층별로 그룹핑하기
def get_direction(wall):
    n = wall.outwardNormal()
    angle = math.degrees(math.atan2(n.x(), n.y()))
    if angle < 0:
        angle += 360
    return round(angle, 1)  # 0=북, 90=동, 180=남, 270=서

# 층 정보 가져오기 (스토리 이름 또는 Z좌표)
def get_floor(wall):
    story = wall.space()
    if story.is_initialized():
        s = story.get().buildingStory()
        if s.is_initialized():
            return s.get().nameString()
    # fallback: Z좌표로 층 추정
    return f"Z={round(wall.centroid().z(), 1)}"

# 층 + 방향으로 그룹핑
groups = defaultdict(list)

for surface in model.getSurfaces():
    if surface.surfaceType() != "Wall":
        continue
    floor = get_floor(surface)
    direction = get_direction(surface)
    groups[(floor, direction)].append(surface)

# 출력
for (floor, angle), walls in sorted(groups.items()):
    print(f"\n[{floor}층 | {angle}°] → {len(walls)}개")
    for w in walls:
        print(f"  {w.nameString()}")


[Building Story 1층 | 45.0°] → 10개
  Space 2-Wall 02
  Space 7-Wall 03 Reversed
  Space 1-Wall 03
  Space 5-Wall 02 Reversed
  Space 6-Wall 02 Reversed
  Space 3-Wall 02
  Space 4-Wall 05 Reversed
  Space 4-Wall 04 Reversed
  Space 4-Wall 07 Reversed
  Space 4-Wall 06 Reversed

[Building Story 1층 | 135.0°] → 10개
  Space 2-Wall 01 Reversed
  Space 1-Wall 01 Reversed
  Space 1-Wall 02 Reversed
  Space 7-Wall 02 Reversed
  Space 7-Wall 01 Reversed
  Space 5-Wall 01
  Space 6-Wall 01 Reversed
  Space 4-Wall 03
  Space 4-Wall 09 Reversed
  Space 3-Wall 01

[Building Story 1층 | 225.0°] → 10개
  Space 7-Wall 05
  Space 2-Wall 04 Reversed
  Space 1-Wall 05 Reversed
  Space 1-Wall 06 Reversed
  Space 4-Wall 11 Reversed
  Space 5-Wall 05
  Space 6-Wall 04
  Space 4-Wall 10 Reversed
  Space 4-Wall 01 Reversed
  Space 3-Wall 05 Reversed

[Building Story 1층 | 315.0°] → 10개
  Space 2-Wall 03 Reversed
  Space 3-Wall 03 Reversed
  Space 1-Wall 04
  Space 7-Wall 04
  Space 5-Wall 03 Reversed
  Space 5-W

Thermal zone 재설정

In [13]:
# 각도 계산
def get_angle(wall):                                                                
    n = wall.outwardNormal()
    angle = math.degrees(math.atan2(n.x(), n.y()))                                        
    return angle + 360 if angle < 0 else angle

# 각도를 45° 단위로 스냅                                                 
def snap_direction(angle, step=45):                                              
    return round(angle / step) * step % 360                                            
def get_story(space):                                                               
    s = space.buildingStory()
    return s.get().nameString() if s.is_initialized() else "Unknown"                
def get_exterior_walls(space):                                                      
    return [                                                                        
        s for s in space.surfaces()
        if s.surfaceType() == "Wall"
        and s.outsideBoundaryCondition() == "Outdoors"
        and abs(s.outwardNormal().z()) < 0.3                                        
    ]

# 공간 분류: interior, corner, standard                  
def classify_space(space):                                                          
    """
    반환값:
    ("interior", None)           - 외벽 없음
    ("corner",   None)           - 외벽 방향 2개 이상                             
    ("standard", dominant_angle) - 외벽 방향 1개                                  
    """
    ext_walls = get_exterior_walls(space)                                           
                                                                                    
    if not ext_walls:
        return ("interior", None)
# 면적 가중 방향 집합
    directions = set(                                                               
        snap_direction(get_angle(w))
        for w in ext_walls
    )                                                                               

    if len(directions) >= 2:
        return ("corner", None)
                                                                                    
    return ("standard", directions.pop())

def is_similar_size(a, b, tol=0.2):
    fa, fb = a.floorArea(), b.floorArea()
    return abs(fa - fb) / max(fa, fb) < tol                                            
# ── 분류 ──────────────────────────────────────────────                               
interior_spaces = []                                                                
corner_spaces   = []
standard_groups = defaultdict(list)  # (story, direction) → [spaces]
for space in model.getSpaces():
    kind, angle = classify_space(space)                                             
    story = get_story(space)                                                           
    if kind == "interior":                                                          
        interior_spaces.append(space)
    elif kind == "corner":
        corner_spaces.append(space)
    else:
        standard_groups[(story, angle)].append(space)                                  
# ── 면적 유사도로 standard 그룹 세분화 ────────────────                               
def cluster_by_size(spaces, tol=0.2):                                               
    clusters = []
    for space in spaces:
        placed = False                                                                        
        for cluster in clusters:
            if is_similar_size(cluster[0], space, tol):                             
                cluster.append(space)
                placed = True
                break                                                                         
            if not placed:
                clusters.append([space])                                                
    return clusters

final_groups = []

for (story, angle), spaces in sorted(standard_groups.items()):                            
    for cluster in cluster_by_size(spaces):
        final_groups.append({                                                       
            "type":    "standard",
            "story":   story,
            "angle":   angle,                                                                     
            "spaces":  cluster
        })                                                                          

for space in corner_spaces:
    final_groups.append({
        "type":   "corner",
        "story":  get_story(space),                                                           
        "angle":  None,
        "spaces": [space]                                                           
    })

for space in interior_spaces:
    final_groups.append({
        "type":   "interior",                                                       
        "story":  get_story(space),
        "angle":  None,                                                             
        "spaces": [space]
    })

# ── 결과 출력 ─────────────────────────────────────────                            
print(f"총 {len(final_groups)}개 Zone 그룹\n")                                        
for i, g in enumerate(final_groups, 1):
    direction = f"{g['angle']}°" if g['angle'] is not None else "-"                 
    space_names = ", ".join(s.nameString() for s in g["spaces"])                    
    print(f"[{i:02d}] {g['type']:8s} | {g['story']:5s} | {direction:6s} | {space_names}")   

총 15개 Zone 그룹

[01] corner   | Building Story 2 | -      | Space 12
[02] corner   | Building Story 2 | -      | Space 10
[03] corner   | Building Story 1 | -      | Space 1
[04] corner   | Building Story 2 | -      | Space 11
[05] corner   | Building Story 1 | -      | Space 7
[06] corner   | Building Story 2 | -      | Space 14
[07] corner   | Building Story 3 | -      | Space 17
[08] corner   | Building Story 3 | -      | Space 23
[09] corner   | Building Story 1 | -      | Space 5
[10] corner   | Building Story 3 | -      | Space 15
[11] corner   | Building Story 3 | -      | Space 21
[12] corner   | Building Story 2 | -      | Space 8
[13] corner   | Building Story 1 | -      | Space 4
[14] corner   | Building Story 1 | -      | Space 3
[15] interior | Building Story 3 | -      | Space 19


In [14]:
# 기존 Thermal Zone 전부 제거
for zone in model.getThermalZones():
    zone.remove()

# final_groups 기반으로 새 Thermal Zone 생성 및 Space 할당
for i, g in enumerate(final_groups, 1):
    new_zone = openstudio.model.ThermalZone(model)
    story = g["story"].replace(" ", "_")
    if g["type"] == "standard":
        zone_name = f"Zone-{story}-{g['type']}-{i:02d}"
    else:
        zone_name = f"Zone-{story}-{g['type']}-{i:02d}"
    new_zone.setName(zone_name)
    for space in g["spaces"]:
        space.setThermalZone(new_zone)

# final_groups에서 누락된 standard space fallback 처리
standard_counter = 1
for space in model.getSpaces():
    if not space.thermalZone().is_initialized():
        kind, angle = classify_space(space)
        story = get_story(space).replace(" ", "_")
        new_zone = openstudio.model.ThermalZone(model)
        zone_name = f"Zone-{story}-{kind}-{standard_counter:02d}"
        new_zone.setName(zone_name)
        space.setThermalZone(new_zone)
        print(f"Fallback zone 생성: {space.nameString()} → {zone_name}")
        standard_counter += 1

print(f"\nThermal Zone 재설정 완료. 총 {len(model.getThermalZones())}개 Zone 생성.")

Fallback zone 생성: Space 6 → Zone-Building_Story_1-standard-01
Fallback zone 생성: Space 20 → Zone-Building_Story_3-standard-02
Fallback zone 생성: Space 2 → Zone-Building_Story_1-standard-03
Fallback zone 생성: Space 16 → Zone-Building_Story_3-standard-04
Fallback zone 생성: Space 18 → Zone-Building_Story_3-standard-05
Fallback zone 생성: Space 22 → Zone-Building_Story_3-standard-06
Fallback zone 생성: Space 13 → Zone-Building_Story_2-standard-07
Fallback zone 생성: Space 9 → Zone-Building_Story_2-standard-08

Thermal Zone 재설정 완료. 총 23개 Zone 생성.


In [15]:
# Zone 미할당 Space 및 분류 확인
for space in model.getSpaces():
    kind, angle = classify_space(space)
    has_zone = space.thermalZone().is_initialized()
    zone_name = space.thermalZone().get().nameString() if has_zone else "없음"
    print(f"{space.nameString():15s} | {kind:8s} | zone: {zone_name}")

Space 12        | corner   | zone: Zone-Building_Story_2-corner-01
Space 6         | standard | zone: Zone-Building_Story_1-standard-01
Space 20        | standard | zone: Zone-Building_Story_3-standard-02
Space 2         | standard | zone: Zone-Building_Story_1-standard-03
Space 10        | corner   | zone: Zone-Building_Story_2-corner-02
Space 1         | corner   | zone: Zone-Building_Story_1-corner-03
Space 11        | corner   | zone: Zone-Building_Story_2-corner-04
Space 7         | corner   | zone: Zone-Building_Story_1-corner-05
Space 14        | corner   | zone: Zone-Building_Story_2-corner-06
Space 17        | corner   | zone: Zone-Building_Story_3-corner-07
Space 16        | standard | zone: Zone-Building_Story_3-standard-04
Space 23        | corner   | zone: Zone-Building_Story_3-corner-08
Space 18        | standard | zone: Zone-Building_Story_3-standard-05
Space 5         | corner   | zone: Zone-Building_Story_1-corner-09
Space 22        | standard | zone: Zone-Building_Sto

In [16]:
for space in model.getSpaces():
    if not space.thermalZone().is_initialized():                                                                                                        
        print(f"Zone 없음: {space.nameString()}")  

In [17]:
for space in model.getSpaces():
    kind, angle = classify_space(space)
    has_zone = space.thermalZone().is_initialized()
    zone_name = space.thermalZone().get().nameString() if has_zone else "없음"
    print(f"{space.nameString():15s} | {kind:8s} | zone: {zone_name}")

Space 12        | corner   | zone: Zone-Building_Story_2-corner-01
Space 6         | standard | zone: Zone-Building_Story_1-standard-01
Space 20        | standard | zone: Zone-Building_Story_3-standard-02
Space 2         | standard | zone: Zone-Building_Story_1-standard-03
Space 10        | corner   | zone: Zone-Building_Story_2-corner-02
Space 1         | corner   | zone: Zone-Building_Story_1-corner-03
Space 11        | corner   | zone: Zone-Building_Story_2-corner-04
Space 7         | corner   | zone: Zone-Building_Story_1-corner-05
Space 14        | corner   | zone: Zone-Building_Story_2-corner-06
Space 17        | corner   | zone: Zone-Building_Story_3-corner-07
Space 16        | standard | zone: Zone-Building_Story_3-standard-04
Space 23        | corner   | zone: Zone-Building_Story_3-corner-08
Space 18        | standard | zone: Zone-Building_Story_3-standard-05
Space 5         | corner   | zone: Zone-Building_Story_1-corner-09
Space 22        | standard | zone: Zone-Building_Sto

## AirLoop HVAC 시스템 재설정

In [18]:
# 모든 AirLoop HVAC 시스템 출력
for loop in model.getAirLoopHVACs():
    zones_in_loop = [
        comp.to_ThermalZone().get().nameString()
        for comp in loop.demandComponents()
        if comp.to_ThermalZone().is_initialized()
    ]
    print(f"{loop.nameString()}")
    for z in zones_in_loop:
        print(f"  └ {z}")

VAV with Reheat
VAV with Reheat 1
VAV with Reheat 2


In [19]:
# AirLoop와 BuildingStory를 이름 기준 정렬 후 순서대로 매핑
air_loops = sorted(model.getAirLoopHVACs(), key=lambda l: l.nameString())
stories    = sorted(model.getBuildingStorys(), key=lambda s: s.nameString())

print("Story → AirLoop 매핑:")
story_to_loop = {}
for loop, story in zip(air_loops, stories):
    story_to_loop[story.nameString()] = loop
    print(f"  {story.nameString()} → {loop.nameString()}")
print()

# 새 VAV Reheat terminal 생성 (hot water 코일 포함)
def make_vav_terminal(model, air_loop):
    # 기존 terminal이 있으면 clone
    for comp in air_loop.demandComponents():
        if comp.iddObjectType().valueDescription() == "OS:AirTerminal:SingleDuct:VAV:Reheat":
            return comp.to_AirTerminalSingleDuctVAVReheat().get().clone(model).to_AirTerminalSingleDuctVAVReheat().get()
    # 없으면 새로 생성 (electric reheat coil)
    reheat_coil = openstudio.model.CoilHeatingElectric(model)
    terminal = openstudio.model.AirTerminalSingleDuctVAVReheat(model, model.alwaysOnDiscreteSchedule(), reheat_coil)
    return terminal

# 미연결 zone을 동일 층 AirLoop에 추가
added, skipped = 0, 0
for zone in model.getThermalZones():
    if zone.airLoopHVAC().is_initialized():
        skipped += 1
        continue

    zone_story = None
    for space in zone.spaces():
        s = space.buildingStory()
        if s.is_initialized():
            zone_story = s.get().nameString()
            break

    if zone_story and zone_story in story_to_loop:
        loop = story_to_loop[zone_story]
        terminal = make_vav_terminal(model, loop)
        loop.addBranchForZone(zone, terminal)
        print(f"추가: {zone.nameString():40s} → {loop.nameString()}")
        added += 1
    else:
        print(f"매핑 없음: {zone.nameString()} (story: {zone_story})")

print(f"\n완료 - 추가: {added}개, 기존 연결: {skipped}개")


Story → AirLoop 매핑:
  Building Story 1 → VAV with Reheat
  Building Story 2 → VAV with Reheat 1
  Building Story 3 → VAV with Reheat 2

추가: Zone-Building_Story_2-corner-01          → VAV with Reheat 1
추가: Zone-Building_Story_2-standard-08        → VAV with Reheat 1
추가: Zone-Building_Story_3-corner-07          → VAV with Reheat 2
추가: Zone-Building_Story_2-corner-06          → VAV with Reheat 1
추가: Zone-Building_Story_2-corner-02          → VAV with Reheat 1
추가: Zone-Building_Story_1-standard-03        → VAV with Reheat
추가: Zone-Building_Story_1-corner-09          → VAV with Reheat
추가: Zone-Building_Story_1-corner-05          → VAV with Reheat
추가: Zone-Building_Story_1-corner-03          → VAV with Reheat
추가: Zone-Building_Story_2-corner-04          → VAV with Reheat 1
추가: Zone-Building_Story_3-corner-08          → VAV with Reheat 2
추가: Zone-Building_Story_3-corner-10          → VAV with Reheat 2
추가: Zone-Building_Story_3-corner-11          → VAV with Reheat 2
추가: Zone-Building_Story_2-c

각 Zone에 Cooling/Heating Thermostat Schedule 추가

In [20]:
# Medium Office ClgSetp, HtgSetp 스케줄 찾기
clg_schedule = None
htg_schedule = None
for sch in model.getSchedules():
    name = sch.nameString()
    if "Medium Office ClgSetp" in name:
        clg_schedule = sch
    if "Medium Office HtgSetp" in name:
        htg_schedule = sch

if clg_schedule is None or htg_schedule is None:
    print("스케줄을 찾을 수 없습니다. 모델에 포함된 스케줄 목록:")
    for sch in model.getSchedules():
        print(f"  {sch.nameString()}")
else:
    count = 0
    for zone in model.getThermalZones():
        thermostat = openstudio.model.ThermostatSetpointDualSetpoint(model)
        thermostat.setCoolingSetpointTemperatureSchedule(clg_schedule)
        thermostat.setHeatingSetpointTemperatureSchedule(htg_schedule)
        zone.setThermostatSetpointDualSetpoint(thermostat)
        count += 1
    print(f"Thermostat 설정 완료: {count}개 Zone")
    print(f"  Cooling: {clg_schedule.nameString()}")
    print(f"  Heating: {htg_schedule.nameString()}")


Thermostat 설정 완료: 23개 Zone
  Cooling: Medium Office ClgSetp
  Heating: Medium Office HtgSetp


In [21]:
# 각 Zone에 Sizing:Zone 설정 추가 (없으면 System Sizing 오류 발생)
count = 0
for zone in model.getThermalZones():
    sizing = zone.sizingZone()
    sizing.setCoolingDesignAirFlowMethod("DesignDay")
    sizing.setHeatingDesignAirFlowMethod("DesignDay")
    sizing.setZoneCoolingDesignSupplyAirTemperature(12.8)
    sizing.setZoneHeatingDesignSupplyAirTemperature(40.0)
    sizing.setZoneCoolingDesignSupplyAirHumidityRatio(0.0085)
    sizing.setZoneHeatingDesignSupplyAirHumidityRatio(0.008)
    count += 1

print(f"Sizing:Zone 설정 완료: {count}개 Zone")


Sizing:Zone 설정 완료: 23개 Zone


## wall, floor, roofceiling material 변경

In [22]:
# 외벽에 사용된 Construction 목록 출력
ext_wall_constructions = set()
for surface in model.getSurfaces():
    if (surface.surfaceType() == "Wall"
            and surface.outsideBoundaryCondition() == "Outdoors"):
        c = surface.construction()
        if c.is_initialized():
            ext_wall_constructions.add(c.get().nameString())

for name in sorted(ext_wall_constructions):
    print(name)

con-r-13woodframewall


In [23]:
# 내벽에 사용된 Construction 목록 출력
ext_wall_constructions = set()
for surface in model.getSurfaces():
    if (surface.surfaceType() == "Wall"
            and surface.outsideBoundaryCondition() != "Outdoors"
            and surface.outsideBoundaryCondition() != "Ground"):
        c = surface.construction()
        if c.is_initialized():
            ext_wall_constructions.add(c.get().nameString())

for name in sorted(ext_wall_constructions):
    print(name)

con-passivefloornoinsulationcarpetorhardwood
con-passivefloornoinsulationcarpetorhardwood Reversed
con-r-0metalframewall
{7b41fd02-d041-4494-9287-8e67011e5864}


In [24]:
# 모든 내벽 construction을 con-r-0metalframewall로 통일
target_construction = None
for c in model.getConstructions():
    if c.nameString() == "con-r-0metalframewall":
        target_construction = c
        break

if target_construction is None:
    print("con-r-0metalframewall을 찾을 수 없습니다.")
else:
    count = 0
    for surface in model.getSurfaces():
        if (surface.surfaceType() == "Wall"
                and surface.outsideBoundaryCondition() != "Outdoors"
                and surface.outsideBoundaryCondition() != "Ground"):
            current = surface.construction()
            if current.is_initialized() and current.get().nameString() != "con-r-0metalframewall":
                surface.setConstruction(target_construction)
                count += 1
    print(f"교체 완료: {count}개 내벽 surface → con-r-0metalframewall")


교체 완료: 6개 내벽 surface → con-r-0metalframewall


In [25]:
# 기존 DesignDay 전부 삭제
for dd in model.getDesignDays():
    dd.remove()
print(f"기존 DesignDay 삭제 완료")

# 서울 여름 설계일 (ASHRAE, Cooling 0.4%)
summer_dd = openstudio.model.DesignDay(model)
summer_dd.setName("Seoul Ann Clg .4% Condns DB=>MWB")
summer_dd.setDayType("SummerDesignDay")
summer_dd.setMonth(7)
summer_dd.setDayOfMonth(21)
summer_dd.setMaximumDryBulbTemperature(33.0)
summer_dd.setDailyDryBulbTemperatureRange(8.4)
summer_dd.setHumidityIndicatingType("Wetbulb")
summer_dd.setHumidityIndicatingConditionsAtMaximumDryBulb(25.5)
summer_dd.setBarometricPressure(101325)
summer_dd.setWindSpeed(2.4)
summer_dd.setWindDirection(225)
summer_dd.setSkyClearness(1.0)
summer_dd.setRainIndicator(False)
summer_dd.setSnowIndicator(False)
summer_dd.setSolarModelIndicator("ASHRAEClearSky")

# 서울 겨울 설계일 (ASHRAE, Heating 99.6%)
winter_dd = openstudio.model.DesignDay(model)
winter_dd.setName("Seoul Ann Htg 99.6% Condns DB")
winter_dd.setDayType("WinterDesignDay")
winter_dd.setMonth(1)
winter_dd.setDayOfMonth(21)
winter_dd.setMaximumDryBulbTemperature(-14.7)
winter_dd.setDailyDryBulbTemperatureRange(0.0)
winter_dd.setHumidityIndicatingType("Wetbulb")
winter_dd.setHumidityIndicatingConditionsAtMaximumDryBulb(-14.7)
winter_dd.setBarometricPressure(101325)
winter_dd.setWindSpeed(6.9)
winter_dd.setWindDirection(270)
winter_dd.setSkyClearness(0.0)
winter_dd.setRainIndicator(False)
winter_dd.setSnowIndicator(True)
winter_dd.setSolarModelIndicator("ASHRAEClearSky")

print("서울 DesignDay 추가 완료")
print(f"  여름: {summer_dd.nameString()} (최고기온 {summer_dd.maximumDryBulbTemperature()}°C)")
print(f"  겨울: {winter_dd.nameString()} (최저기온 {winter_dd.maximumDryBulbTemperature()}°C)")


기존 DesignDay 삭제 완료
서울 DesignDay 추가 완료
  여름: Seoul Ann Clg .4% Condns DB=>MWB (최고기온 33.0°C)
  겨울: Seoul Ann Htg 99.6% Condns DB (최저기온 -14.7°C)


In [26]:
# SimulationControl - Zone Sizing 활성화
sim_control = model.getSimulationControl()
sim_control.setDoZoneSizingCalculation(True)
sim_control.setDoSystemSizingCalculation(True)
sim_control.setDoPlantSizingCalculation(True)

print("SimulationControl 설정 완료")
print(f"  Zone Sizing  : {sim_control.doZoneSizingCalculation()}")
print(f"  System Sizing: {sim_control.doSystemSizingCalculation()}")
print(f"  Plant Sizing : {sim_control.doPlantSizingCalculation()}")


SimulationControl 설정 완료
  Zone Sizing  : True
  System Sizing: True
  Plant Sizing : True


In [ ]:
# 저장할 OSM 파일 경로 설정
save_path = Path(r"C:\Users\ryudo\OneDrive\Desktop\SampleBuilding\SampleBuilding_final.osm")

# 새로운 이름으로 저장
model.save(openstudio.path(str(save_path)), True)

print("저장 완료:", save_path)

저장 완료: C:\Users\ryudo\OneDrive\Desktop\SampleBuilding\SampleBuilding_changed.osm


## 시뮬레이션

In [ ]:
import subprocess, sqlite3
from pathlib import Path

os_exe   = r"C:\openstudioapplication-1.11.0\bin\openstudio.exe"
osw_path = Path(r"C:\Users\ryudo\OneDrive\Desktop\SampleBuilding\SampleBuilding_final\workflow.osw")
sql_path = osw_path.parent / "run" / "eplusout.sql"

# 시뮬레이션 실행
print("시뮬레이션 실행 중...")
result = subprocess.run(
    [os_exe, "run", "-w", str(osw_path)],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("[STDERR]", result.stderr[-1000:])
    raise RuntimeError("시뮬레이션 실패")

print("\n시뮬레이션 완료")

시뮬레이션 실행 중...


시뮬레이션 완료

=== 연간 에너지 소비량 ===
  Total Site Energy              Total Energy              1772.49 GJ
  Net Site Energy                Total Energy              1772.49 GJ
  Total Source Energy            Total Energy              5003.87 GJ
  Net Source Energy              Total Energy              5003.87 GJ
  Total Site Energy              Energy Per Total Building Area       498.15 MJ/m2
  Net Site Energy                Energy Per Total Building Area       498.15 MJ/m2
  Total Source Energy            Energy Per Total Building Area      1406.32 MJ/m2
  Net Source Energy              Energy Per Total Building Area      1406.32 MJ/m2
  Total Site Energy              Energy Per Conditioned Building Area       498.15 MJ/m2
  Net Site Energy                Energy Per Conditioned Building Area       498.15 MJ/m2
  Total Source Energy            Energy Per Conditioned Building Area      1406.32 MJ/m2
  Net Source Energy              Energy Per Conditioned Building Area      1

In [39]:
# 결과 추출
conn = sqlite3.connect(sql_path)
cur  = conn.cursor()

def safe_float(v):
    try: return float(v)
    except: return None

print("\n=== Site and Source Energy ===")
cur.execute("""
    SELECT RowName, ColumnName, Units, Value
    FROM TabularDataWithStrings
    WHERE ReportName = 'AnnualBuildingUtilityPerformanceSummary'
      AND TableName  = 'Site and Source Energy'
""")
for row in cur.fetchall():
    v = safe_float(row[3])
    if v is not None:
        print(f"  {row[0]:40s} {row[1]:25s} {v:>12.2f} {row[2]}")

print("\n=== End Uses ===")
cur.execute("""
    SELECT RowName, ColumnName, Units, Value
    FROM TabularDataWithStrings
    WHERE ReportName = 'AnnualBuildingUtilityPerformanceSummary'
      AND TableName  = 'End Uses'
""")
for row in cur.fetchall():
    v = safe_float(row[3])
    if v is not None and v > 0:
        print(f"  {row[0]:30s} {row[1]:20s} {v:>12.2f} {row[2]}")

conn.close()



=== Site and Source Energy ===
  Total Site Energy                        Total Energy                   1772.49 GJ
  Net Site Energy                          Total Energy                   1772.49 GJ
  Total Source Energy                      Total Energy                   5003.87 GJ
  Net Source Energy                        Total Energy                   5003.87 GJ
  Total Site Energy                        Energy Per Total Building Area       498.15 MJ/m2
  Net Site Energy                          Energy Per Total Building Area       498.15 MJ/m2
  Total Source Energy                      Energy Per Total Building Area      1406.32 MJ/m2
  Net Source Energy                        Energy Per Total Building Area      1406.32 MJ/m2
  Total Site Energy                        Energy Per Conditioned Building Area       498.15 MJ/m2
  Net Site Energy                          Energy Per Conditioned Building Area       498.15 MJ/m2
  Total Source Energy                      Energy Per Cond

In [ ]:
# 시뮬레이션 결과 SQL 파일에서 테이블 목록 출력
import sqlite3                                                                                                                                
from pathlib import Path

sql_path = Path(r"C:\Users\ryudo\OneDrive\Desktop\SampleBuilding\SampleBuilding_changed\run\eplusout.sql")                                      
conn = sqlite3.connect(sql_path)
cur = conn.cursor()                                                                                                                           

cur.execute("SELECT DISTINCT ReportName, TableName FROM TabularDataWithStrings LIMIT 50")                                                       
for row in cur.fetchall():
    print(row)                                                                                                                                
conn.close() 

('AnnualBuildingUtilityPerformanceSummary', 'Site and Source Energy')
('AnnualBuildingUtilityPerformanceSummary', 'Site to Source Energy Conversion Factors')
('AnnualBuildingUtilityPerformanceSummary', 'Building Area')
('AnnualBuildingUtilityPerformanceSummary', 'End Uses')
('AnnualBuildingUtilityPerformanceSummary', 'End Uses By Subcategory')
('AnnualBuildingUtilityPerformanceSummary', 'End Uses By Space Type')
('AnnualBuildingUtilityPerformanceSummary', 'Utility Use Per Conditioned Floor Area')
('AnnualBuildingUtilityPerformanceSummary', 'Utility Use Per Total Floor Area')
('AnnualBuildingUtilityPerformanceSummary', 'Electric Loads Satisfied')
('AnnualBuildingUtilityPerformanceSummary', 'On-Site Thermal Sources')
('AnnualBuildingUtilityPerformanceSummary', 'Water Source Summary')
('AnnualBuildingUtilityPerformanceSummary', 'Setpoint Not Met Criteria')
('AnnualBuildingUtilityPerformanceSummary', 'Comfort and Setpoint Not Met Summary')
('InputVerificationandResultsSummary', 'General')


In [29]:
import openstudio                                                                                                                               
print(openstudio.__file__)

c:\Users\ryudo\miniconda3\envs\eplus\Lib\site-packages\openstudio\__init__.py


In [36]:
import subprocess                                                                                                                               
result = subprocess.run(["where", "energyplus"], capture_output=True, text=True, shell=True)                                                  
print(result.stdout)                                                                                                                                                        